In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/activity_labels.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/README.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/features_info.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/features.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/subject_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/X_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_acc_y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/total_acc_y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/total_acc_z_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_acc_z_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_gyr

# Loading the Dataset

In [2]:
import numpy as np

def load_raw_signals(folder_path, filenames): # a function to load the dataset
    signals = []
    for name in filenames:
        file_path = f"{folder_path}/{name}.txt" # makes the file path fro every file
        data = np.loadtxt(file_path) #Loads the data from the file into an Numpy array
        signals.append(data) # appends the list we created with the new readings
    
    return np.transpose(np.array(signals), (1, 2, 0)) # for our project it requires we have the array in (sample, step, feature) format

# file names we want to load
filenames = [
    'body_acc_x_train', 'body_acc_y_train', 'body_acc_z_train',
    'body_gyro_x_train', 'body_gyro_y_train', 'body_gyro_z_train'
]

#The absolut path for the folder in our dataset where the files are located
raw_data_path = '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/Inertial Signals'
X_raw = load_raw_signals(raw_data_path, filenames)

print(f"Raw 6-axis data shape: {X_raw.shape}") #printing the shape

Raw 6-axis data shape: (7352, 128, 6)


# Normalizing the data

In [3]:
import numpy as np

# --- STEP 1: Load everything first ---
def load_raw_signals(folder_path, filenames):
    signals = []
    for name in filenames:
        file_path = f"{folder_path}/{name}.txt"
        data = np.loadtxt(file_path)
        signals.append(data)
    # Returns (samples, 128, 6)
    return np.transpose(np.array(signals), (1, 2, 0))
filenames = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z'
]
filenames_train = [f + '_train' for f in filenames]
filenames_test = [f + '_test' for f in filenames]
raw_data_path_train = '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/Inertial Signals'
raw_data_path_test =  '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals'

x_train_raw = load_raw_signals(raw_data_path_train, filenames_train)
X_test_raw = load_raw_signals(raw_data_path_test, filenames_test)

# --- STEP 2: Calculate Normalization ONLY on Training Data ---
# This prevents "Data Leakage"
mean = np.mean(x_train_raw, axis=(0, 1))
std = np.std(x_train_raw, axis=(0, 1))

# --- STEP 3: Normalize BOTH sets using Training constants ---
x_train = (x_train_raw - mean) / (std + 1e-8)
X_test = (X_test_raw - mean) / (std + 1e-8) # This was the missing step!

# --- STEP 4: Load Labels ---
y_train = np.loadtxt('/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/y_train.txt', dtype=int) - 1
y_test = np.loadtxt('/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/y_test.txt', dtype=int) - 1

print(f"Normalized x_train shape: {x_train.shape}")
print(f"Normalized X_test shape: {X_test.shape}")

Normalized x_train shape: (7352, 128, 6)
Normalized X_test shape: (2947, 128, 6)


# Coding the 1D-CNN Architecture

In [4]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers, models
model = keras.Sequential(name="HAR_1DCNN") 
model.add(layers.Input(shape=(128, 6))) # loading an empty stack to which we'll add layers
model.add(layers.Conv1D(   #Now we're adding layers to this 
    filters=64,    #this arg defines how many local features we'll look for
    kernel_size=5, #this is essentially our window size which we slide over our 128 timesteps to extract features
    activation='relu', #rectified linear unit introduces non-linearity basically acting as a gate to decide which information is important enough to be passed to the next layer
    padding= 'same',   #This makes sure that our output is the same length as our input adds zeros so the kernel can center itself
    #input_shape=(128,6) #This defines the size of our input passed 
))
model.add(layers.MaxPooling1D(pool_size=2))
model.add(layers.Dropout(rate=0.5)) 

model.add(layers.Conv1D(   
    filters=128,   
    kernel_size=3, 
    activation='relu', 
    padding= 'same',  
))
model.add(layers.MaxPooling1D(pool_size=2))
model.add(layers.Dropout(rate=0.5))
model.add(layers.Flatten())

# 1. Add a "Brain" (Hidden Layer)
model.add(layers.Dense(128, activation='relu')) 

# 2. Add another Dropout here to stop the Dense layer from over-memorizing
model.add(layers.Dropout(0.5))


model.add(layers.Dense(6, activation='softmax'))
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_test, y_test),
    shuffle=True,
)

2026-05-06 16:08:19.189296: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778083699.602965      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778083699.720659      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778083700.706832      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778083700.706878      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778083700.706881      23 computation_placer.cc:177] computation placer alr

Model: "HAR_1DCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 128, 64)        │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 64, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 551,878 (2.11 MB)

 Trainable params: 551,878 (2.11 MB)

...

Epoch 1/30


I0000 00:00:1778083743.454212      70 service.cc:152] XLA service 0x7a9c0400b880 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778083743.454244      70 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778083743.454248      70 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778083743.880321      70 cuda_dnn.cc:529] Loaded cuDNN version 91002


 55/460 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.1840 - loss: 1.8842

I0000 00:00:1778083747.823998      70 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


460/460 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.3515 - loss: 1.3613 - val_accuracy: 0.6566 - val_loss: 0.6651
Epoch 2/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6449 - loss: 0.6597 - val_accuracy: 0.7061 - val_loss: 0.6112
Epoch 3/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6885 - loss: 0.5890 - val_accuracy: 0.6759 - val_loss: 0.5715
Epoch 4/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7185 - loss: 0.5570 - val_accuracy: 0.7920 - val_loss: 0.5082
Epoch 5/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7861 - loss: 0.4754 - val_accuracy: 0.8113 - val_loss: 0.4549
Epoch 6/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8161 - loss: 0.4163 - val_accuracy: 0.8833 - val_loss: 0.3455
Epoch 7/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8328 - loss: 0.3914 - val_accuracy: 0.8609 - val_loss: 0.3688
Epoch 8/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8592 - loss: 0.3426 - val_accuracy: 0.8670 - va